In [ ]:
"""
Satellite Image Classification using Random Forest

This script processes satellite imagery for land cover classification using Random Forest.
It handles multi-band satellite images/tiles, trains a model with ground truth data, and generates 
classified maps with probability scores.

Note: Satellite, training, and mask data tiles share the same name and ID.
"""

In [ ]:
from osgeo import gdal, gdal_array
import os
import glob
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
%matplotlib inline

# Configure GDAL settings
gdal.UseExceptions()
gdal.AllRegister()

In [ ]:
# Read satellite image and training data paths
dir_sat = r"your_path\\"
dir_train = r"your_path\\"

# Count number of tiles to process
count = 0
for path in os.listdir(dir_sat):
    if os.path.isfile(os.path.join(dir_sat, path)):
        count += 1
print('Number of tiles to process: ', count)

# Read tiles and prepare dataframes
with rasterio.open(r"yourpath\satellite_data_sample.tif") as data:
    bands = data.count

df_img = pd.DataFrame(dtype='float', columns=['band' + str(x) for x in range(1,bands+1)])
df_train = pd.DataFrame(dtype='float', columns=['Training_Data'])
print('Done!')

# Process satellite image tiles
for filename in glob.iglob(f'{dir_sat}//*.tif'):
    with rasterio.open(filename) as img_ds:
        tmp = np.zeros((img_ds.height*img_ds.width, img_ds.count))
        for b in range(img_ds.count):
            tmp[:, b] = img_ds.read(b+1).flatten()
        df_tmp = pd.DataFrame(tmp, columns=['band' + str(x) for x in range(1,b+2)])
    df_img = pd.concat([df_img, df_tmp], axis = 0)

# Process training data tiles    
for filename in glob.iglob(f'{dir_train}//*.tif'):
    with rasterio.open(filename) as roi:
        tmp_roi = np.zeros((roi.height*roi.width, roi.count))
        tmp_roi[:, 0] = roi.read(1, out_dtype=None).flatten() # out_dtype=None preserves original data values
        df_roi = pd.DataFrame(tmp_roi, columns=['Training_Data'])
        df_roi['Training_Data'] = df_roi['Training_Data'].replace(roi.nodatavals[0], np.nan)
    df_train = pd.concat([df_train, df_roi], axis = 0)

# Combine and clean data
df = pd.concat([df_img, df_train], axis=1)
df = df[df['Training_Data'].notna()]
df = df.drop_duplicates()
df.to_excel("output_excel.xlsx") 

In [ ]:
# Split data into features and labels
features = df.drop(columns=['Training_Data'], axis=1) 
labels = df['Training_Data']

# Split into training and testing sets
from sklearn.model_selection import train_test_split
train_features, test_features, train_labels, test_labels = train_test_split(
    features, labels, test_size=0.30, random_state=42
)

print('Training Features Shape:', train_features.shape)
print('Training Labels Shape:', train_labels.shape)
print('Testing Features Shape:', test_features.shape)
print('Testing Labels Shape:', test_labels.shape)

In [ ]:
# Initialize and tune Random Forest model
from sklearn.ensemble import RandomForestClassifier

# Initialize base model
rf = RandomForestClassifier(random_state=42)

# Define hyperparameter search space
n_estimators = [int(x) for x in np.linspace(start=200, stop=2000, num=5)]
max_features = ['log2', 'sqrt']
max_depth = [int(x) for x in np.linspace(5, 25, num=5)]
min_samples_split = [2,5,10,15]
min_samples_leaf = [1, 2, 4, 10]
bootstrap = [True, False]

hyperF = dict(
    n_estimators=n_estimators,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    max_features=max_features,
    bootstrap=bootstrap
)

# Perform RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV
import dask_ml.model_selection as dcv

randomF = dcv.RandomizedSearchCV(
    rf, hyperF, n_iter=20, n_jobs=-1, cv=3, random_state=42
)

In [ ]:
# Train model
bestF = randomF.fit(train_features, train_labels)
print("The mean accuracy of the model is:", bestF.score(test_features, test_labels))

# Save trained model
joblib.dump(bestF, "./RF_model.joblib")

# Display best parameters
print("Best parameters:", bestF.best_estimator_)

# Check cross-validation results
cv_results = pd.DataFrame(bestF.cv_results_)
print("\nCross-validation results:")
print(cv_results)

# Evaluate model performance
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def evaluate(model, test_features, test_labels):
    predictions = model.predict(test_features)
    print('\nConfusion Matrix:', confusion_matrix(test_labels, predictions))
    print('\nClassification Report:')
    print(classification_report(test_labels, predictions))
    print("\nAccuracy:", accuracy_score(test_labels, predictions) * 100) 
    return predictions

predictions = evaluate(bestF, test_features, test_labels)

In [ ]:
# Visualize confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels_1 = {
    '3':'Deciduous fruit tree', '4':'Evergreen fruit tree', 
    '5':'Hardwood plantation', '6':'Softwood plantation',
    '7':'Pastures and grassland', '8':'Cereals', 
    '9':'Legumes', '10':'Oilseeds',
    '11':'Vegetable and herbs', '12':'Native trees and shrubland'
}

label_temp = list(range(3,13))
labels_list = [labels_1[key] for key in labels_1]

cm = confusion_matrix(test_labels, predictions, labels=label_temp)
cmd = ConfusionMatrixDisplay(cm, display_labels=labels_1.values())
cmd.plot(xticks_rotation='vertical', cmap='Greys')
plt.tight_layout()

# Display feature importance
bands = list(range(31))
for b, imp in zip(bands, bestF.best_estimator_.feature_importances_):
    print(f'Band {b} importance: {round(imp, 2)}')

# Classify new satellite imagery
dir_img = r"your_path" # satellite tiles
dir_mask = r"your_path" # mask tiles
dir_class = r"your_path\\" # classified tiles
dir_prob = r"your_path\\" # probability tiles

# Load saved model
loaded_rf = joblib.load("./RF_model.joblib")
modelname = "RF"

# Process each image tile
for filename in glob.iglob(f'{dir_img}//*.tif'):
    with rasterio.open(filename) as img:
        # Prepare image data
        arr = np.zeros((img.height*img.width, img.count))
        for b in range(img.count):
            arr[:, b] = img.read(b+1).flatten()
        df_arr = pd.DataFrame(arr, columns=['band' + str(x) for x in range(1,b+2)])
        
        # Find and process mask
        name = 't' + filename[-8:]
        filename_msk = find(name, dir_mask)
        
        with rasterio.open(filename_msk) as msk:
            mask = np.zeros((msk.height*msk.width, msk.count))
            mask[:, 0] = msk.read(1).flatten()
            df_msk = pd.DataFrame(mask, columns=['mask'])
            df_msk['mask'] = df_msk['mask'].replace(msk.nodatavals[0], np.nan)
        
        df_img = pd.concat([df_arr, df_msk], axis=1)
        df_img.dropna(inplace=True)
        
        ini_data = df_img.drop(columns=['mask'], axis=1) 
        ind = ini_data.index.tolist()
        
        if ini_data.shape[0] != 0:
            # Generate predictions and probabilities
            class_prediction = loaded_rf.predict(ini_data)
            df_class = pd.DataFrame(class_prediction, columns=['predictions'], index=ind)
            df_out = pd.concat([df_msk, df_class], axis=1).drop(columns=['mask'])
            out = df_out['predictions'].tolist()
        else:
            out = np.repeat([[np.nan]]*msk.height, msk.width, axis=1).tolist()
            
        # Save classified image
        class_pred = np.array(out).reshape(img.shape)
        imgname = os.path.basename(filename).split('.')[0]
        path = dir_class + imgname + "_" + modelname + ".tif"
        
        kwargs = msk.meta
        kwargs['crs'] = img.meta['crs']
        
        with rasterio.open(path, 'w', **kwargs) as dst:
            dst.write_band(1, class_pred)
    
        print(f"Processed: {path}")

# Calculate class probabilities
for filename in glob.iglob(f'{dir_img}//*.tif'):
    with rasterio.open(filename) as img:
        # Process image data
        arr = np.zeros((img.height*img.width, img.count))
        for b in range(img.count):
            arr[:, b] = img.read(b+1).flatten()
        df_arr = pd.DataFrame(arr, columns=['band' + str(x) for x in range(1,b+2)])
        
        # Process mask
        name = 't' + filename[-8:]
        filename_msk = find(name, dir_mask)
        
        with rasterio.open(filename_msk) as msk:
            mask = np.zeros((msk.height*msk.width, msk.count))
            mask[:, 0] = msk.read(1).flatten()
            df_msk = pd.DataFrame(mask, columns=['mask'])
            df_msk['mask'] = df_msk['mask'].replace(msk.nodatavals[0], np.nan)
        
        df_img = pd.concat([df_arr, df_msk], axis=1)
        df_img.dropna(inplace=True)
        
        ini_data = df_img.drop(columns=['mask'], axis=1) 
        ind = ini_data.index.tolist()
        
        if ini_data.shape[0] != 0:
            # Calculate probabilities
            class_probability = np.max(loaded_rf.predict_proba(ini_data), axis=1)
            df_prob = pd.DataFrame(class_probability, columns=['probability'], index=ind)
            df_out_p = pd.concat([df_msk, df_prob], axis=1).drop(columns=['mask'])
            prob = df_out_p['probability'].tolist()
        else:
            prob = np.repeat([[np.nan]]*msk.height, msk.width, axis=1).tolist()
            
        # Save probability map
        class_prob = np.array(prob).reshape(img.shape)
        imgname = os.path.basename(filename).split('.')[0]
        path = dir_prob + imgname + "_" + modelname + "_prob.tif"
        
        kwargs = msk.meta
        kwargs['dtype'] = 'float64'
        
        with rasterio.open(path, 'w', **kwargs) as dst:
            dst.write_band(1, class_prob)
    
        print(f"Processed probabilities: {path}")